# 07. Маленькая decoder-only языковая модель

Лекция: [От token IDs до logits](../../notes/interview-prep/08k-tiny-language-model.md).

Сегодня собираем модель, не обучаем. Attention и блок перенесены из твоего завершённого урока 06d без изменения кода. Начни только с A: две строки forward и три ответа.

Ноутбук подготовлен без запуска. Ментор читает твои сохранённые outputs, не запускает проверки.

In [1]:
import math
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(42)


## Готовые компоненты — выполни, не переписывай

Это твои attention и TransformerBlock. Далее добавим только оболочку языковой модели.

In [2]:
class VectorizedCausalAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        if d_model <= 0 or num_heads <= 0 or d_model % num_heads != 0:
            raise ValueError('Нужны положительные D,H и D, кратное H')
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_projection = nn.Linear(d_model, d_model, bias=False)
        self.key_projection = nn.Linear(d_model, d_model, bias=False)
        self.value_projection = nn.Linear(d_model, d_model, bias=False)
        self.output_projection = nn.Linear(d_model, d_model, bias=False)

    def project_heads(self, x):
        if x.ndim != 3 or x.shape[-1] != self.d_model or x.shape[1] == 0:
            raise ValueError('Ожидается непустой x формы (B,T,D)')
        batch_size, length, _ = x.shape
        # Готовая реализация из урока 05c: проекция и разделение голов.
        q, k, v = [
            projection(x)
            .reshape(batch_size, length, self.num_heads, self.head_dim)
            .transpose(1, 2)
            for projection in (
                self.query_projection,
                self.key_projection,
                self.value_projection,
            )
        ]

        return q, k, v

    def forward(self, x):
        batch_size, length, _ = x.shape
        q, k, v = self.project_heads(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(q.shape[-1])
        positions = torch.arange(x.shape[1], device=x.device)
        blocked = positions[None, :] > positions[:, None]
        masked_scores = scores.masked_fill(blocked, float('-inf'))
        weights = torch.softmax(masked_scores, dim=-1)
        context = weights @ v
        combined = context.transpose(1, 2).reshape(batch_size, length, self.d_model)
        return self.output_projection(combined)

In [3]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.norm_1 = nn.LayerNorm(d_model)
        self.attention = VectorizedCausalAttention(d_model, num_heads)
        self.norm_2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        x = x + self.attention(self.norm_1(x))
        x = x + self.mlp(self.norm_2(x))
        return x


## A. От ID до оценок следующего токена

Мини-словарь нужен только для наглядности. V=6 не равно D=8. Не добавляем токенизатор или загрузку корпуса в этот урок.

In [4]:
vocabulary = ['кот', 'ест', 'рыбу', '.', 'пёс', 'спит']
stoi = {token: i for i, token in enumerate(vocabulary)}
vocab_size = len(vocabulary)
full_ids = torch.tensor([[stoi[token] for token in ['кот', 'ест', 'рыбу', '.']]], dtype=torch.long)
inputs = full_ids[:, :-1]  # кот, ест, рыбу
print('Входные ID:', inputs)


Входные ID: tensor([[0, 1, 2]])


В forward заполни только:

1. Сумму token embeddings и positional embeddings (не cat).
2. Вызов выходной проекции self.lm_head после финального LayerNorm.

Блоки, цикл и нормализация уже готовы. Параметры создаём в __init__, используем в forward.

In [5]:
class TinyLanguageModel(nn.Module):
    def __init__(self, vocab_size, max_seq_len=16, d_model=8,
                 num_heads=2, d_ff=32, n_layers=1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff)
            for _ in range(n_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, token_ids):
        if token_ids.ndim != 2:
            raise ValueError('Ожидаются token IDs формы (B,T)')
        length = token_ids.shape[1]
        if not 1 <= length <= self.max_seq_len:
            raise ValueError('Длина должна быть от 1 до max_seq_len')
        positions = torch.arange(length, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.lm_head(x)
        return logits


In [6]:
model = TinyLanguageModel(vocab_size)
logits = model(inputs)
print('Форма входа:', inputs.shape)  # (1,3)
print('Форма logits:', logits.shape)  # (1,3,6)
print('Число параметров:', sum(p.numel() for p in model.parameters()))

with torch.no_grad():
    next_probabilities = torch.softmax(logits[0, -1], dim=-1)
    for token, probability in zip(vocabulary, next_probabilities.tolist()):
        print(f'{token}: {probability:.3f}')
# Модель не обучена. Вероятность точки пока не обязана быть максимальной.


Форма входа: torch.Size([1, 3])
Форма logits: torch.Size([1, 3, 6])
Число параметров: 1080
кот: 0.093
ест: 0.131
рыбу: 0.197
.: 0.246
пёс: 0.191
спит: 0.143


### Остановка A

1. Что означает последняя размерность 6 в logits.shape=(1,3,6)? Почему это не d_model=8?
2. После какого входного контекста предсказывает logits[0,-1]?
3. Почему мы не ожидаем правильного предсказания точки от этой модели прямо сейчас?

**Мои ответы:**

1. У нас логиты вычисляются на весь словарь. Т е для каждого токена в последовательнсти будут посчитаны логиты на каждый токен в словаре. 6 - это размер словаря
2. 'кот', 'ест', 'рыбу'
3. потому что у нас нет следующего за точкой символа

Сохрани и пришли на разбор. B можно делать после A.

**Разбор ментора (исходные ответы сохранены):**

1. Верно: последняя ось — шесть кандидатов из словаря; d_model=8 — ширина внутренних признаков, а не размер словаря.
2. Верно: logits[0,-1] предсказывает продолжение после «кот ест рыбу».
3. Причина другая: модель ещё не обучалась, параметры случайно инициализированы. Мы предсказываем саму точку после «рыбу», а не символ после точки. Точка есть в targets. Отсутствие символа после неё этому примеру не мешает. В сохранённом выводе точка случайно получила наибольшую вероятность 0.246; это не свидетельство обучения или качества.


## B. Цели сдвинуты на один токен — готовый опыт

Прочитай разделы 6–7 лекции. Все три позиции дают обучающий сигнал. На вход loss подаются logits; softmax из A был только для просмотра вероятностей.

In [7]:
targets = full_ids[:, 1:]  # ест, рыбу, .
for position in range(inputs.shape[1]):
    context = [vocabulary[i] for i in inputs[0, :position + 1].tolist()]
    target = vocabulary[targets[0, position].item()]
    print('Контекст:', ' '.join(context), '-> target:', target)

logits = model(inputs)
loss = F.cross_entropy(
    logits.reshape(-1, vocab_size),
    targets.reshape(-1),
)
print('Targets:', targets)
print('Loss:', loss.item())
# backward и optimizer.step отсутствуют: параметры не обновлялись.


Контекст: кот -> target: ест
Контекст: кот ест -> target: рыбу
Контекст: кот ест рыбу -> target: .
Targets: tensor([[1, 2, 3]])
Loss: 1.6867436170578003


### Остановка B

1. Почему target первого токена — «ест», а не «кот»?
2. Почему первая позиция не может подсмотреть «ест» во второй позиции входа?
3. Изменились ли веса после вычисления loss?

**Мои ответы:**

1. Потому что мы предсказываем следующий токен для каждой позиции
2. Потому что мы нигде не смешиваем эту информацию. А в attention мы маскируем будущие токены
3. нет, только лосс посчитали. Градиент не вычисляли

Следующий урок — обучение на небольшом тексте и генерация. Пока мы собрали архитектуру и вычислили ошибку, но не обучили модель.

**Разбор ментора (исходные ответы сохранены):**

Все три ответа верны. Для третьего уточнение: веса обновляет optimizer.step() (либо ручная запись), а backward только вычисляет и накапливает градиенты. Поэтому даже вызов backward сам по себе не изменил бы веса.

По сохранённым outputs: inputs (1,3), logits (1,3,6), targets [[1,2,3]], loss ≈1.68674, параметров 1080. Подсчёт: token embeddings 6*8=48, positional embeddings 16*8=128, блок 840, финальный LayerNorm 16, lm_head 8*6=48. Итого 1080. Это forward и вычисление ошибки необученной модели, не результат обучения.

Код и текстовые outputs прочитаны без выполнения ячеек, запуска проверок и визуального рендеринга по договорённости. Исходные ответы сохранены. Практика сборки модели завершена; следующий этап — обучение и генерация.
